# Wool Quality Classification - Experimentation Notebook

This notebook allows you to interactively experiment with the wool quality classification project.

## Contents
1. Setup and Data Loading
2. Data Exploration
3. Model Training
4. Model Evaluation
5. Predictions

## 1. Setup and Imports

In [ ]:
# Import libraries
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import tensorflow as tf
from tensorflow import keras

# Import project modules
sys.path.append('..')
import config
from src.data_loader import WoolDataLoader
from src.model import WoolClassifier
from src.train import ModelTrainer
from src.evaluate import ModelEvaluator
from src.predict import WoolPredictor

# Set random seeds
np.random.seed(config.RANDOM_SEED)
tf.random.set_seed(config.RANDOM_SEED)

# Configure plotting
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
%matplotlib inline

print("TensorFlow version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))
print("Setup complete!")

## 2. Data Loading and Exploration

In [ ]:
# Load dataset
loader = WoolDataLoader()
X, y, df = loader.load_dataset()

print(f"Total images loaded: {len(X)}")
print(f"Image shape: {X[0].shape}")
print(f"\nDataset info:")
print(df.head())

In [ ]:
# Visualize class distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Overall distribution
df['class'].value_counts().plot(kind='bar', ax=axes[0], color=['red', 'green'])
axes[0].set_title('Overall Class Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

# Distribution by category
category_class = df.groupby(['category', 'class']).size().unstack()
category_class.plot(kind='bar', ax=axes[1], color=['red', 'green'])
axes[1].set_title('Class Distribution by Category', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Category')
axes[1].set_ylabel('Count')
axes[1].legend(title='Class')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print("\nDistribution by Category:")
print(category_class)

In [ ]:
# Visualize sample images
def show_samples(df, X, y, num_samples=16):
    """Display random sample images"""
    fig, axes = plt.subplots(4, 4, figsize=(15, 15))
    
    indices = np.random.choice(len(X), size=num_samples, replace=False)
    
    for idx, ax in enumerate(axes.flat):
        i = indices[idx]
        ax.imshow(X[i])
        
        category = df.iloc[i]['category']
        class_name = df.iloc[i]['class']
        color = 'green' if class_name == 'Good' else 'red'
        
        ax.set_title(f'{category} - {class_name}', color=color, fontweight='bold')
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()

show_samples(df, X, y)

## 3. Data Splitting

In [ ]:
# Split data into train, validation, and test sets
X_train, X_val, X_test, y_train, y_val, y_test = loader.split_data(X, y)

print(f"Training set: {len(X_train)} images")
print(f"Validation set: {len(X_val)} images")
print(f"Test set: {len(X_test)} images")

# Calculate class weights
class_weights = loader.get_class_weights(y_train)

## 4. Model Building

In [ ]:
# Build EfficientNet model (recommended)
model_name = 'efficientnet'  # Change to try different models

classifier = WoolClassifier(model_name=model_name)
model = classifier.build()
classifier.compile()

# Display model summary
classifier.summary()

print(f"\nTotal parameters: {model.count_params():,}")

## 5. Model Training

In [ ]:
# Train model (start with fewer epochs for testing)
trainer = ModelTrainer(model_name=model_name, epochs=10)
history, model_dir = trainer.train(
    X_train, y_train,
    X_val, y_val,
    class_weights=class_weights,
    fine_tune=False
)

print(f"\nModel saved in: {model_dir}")

In [ ]:
# Plot training history
trainer.plot_training_history()

## 6. Model Evaluation

In [ ]:
# Load best model and evaluate
best_model_path = os.path.join(model_dir, 'best_model.keras')
evaluator = ModelEvaluator(best_model_path)

metrics, y_pred, y_pred_proba = evaluator.evaluate(X_test, y_test)

In [ ]:
# Confusion Matrix
evaluator.plot_confusion_matrix(y_test, y_pred)

In [ ]:
# ROC Curve
evaluator.plot_roc_curve(y_test, y_pred_proba)

In [ ]:
# Visualize predictions
evaluator.visualize_predictions(X_test, y_test, y_pred, num_samples=16)

In [ ]:
# Classification report
report = evaluator.classification_report_text(y_test, y_pred)

## 7. Making Predictions

In [ ]:
# Initialize predictor
predictor = WoolPredictor(best_model_path)

In [ ]:
# Predict on a single image
# Replace with your image path
image_path = 'path/to/your/image.jpg'

if os.path.exists(image_path):
    predictor.visualize_prediction(image_path)
else:
    print("Please provide a valid image path")

## 8. Compare Models (Optional)

In [ ]:
# Compare different architectures
results = {}

for model_name in ['cnn_scratch', 'mobilenet', 'efficientnet']:
    print(f"\n{'='*60}")
    print(f"Training: {model_name}")
    print('='*60)
    
    trainer = ModelTrainer(model_name=model_name, epochs=5)
    history, model_dir = trainer.train(
        X_train, y_train, X_val, y_val,
        class_weights=class_weights
    )
    
    # Evaluate
    best_model_path = os.path.join(model_dir, 'best_model.keras')
    evaluator = ModelEvaluator(best_model_path)
    metrics, _, _ = evaluator.evaluate(X_test, y_test)
    
    results[model_name] = metrics

# Compare results
results_df = pd.DataFrame(results).T
print("\nModel Comparison:")
print(results_df)

# Plot comparison
results_df.plot(kind='bar', figsize=(12, 6))
plt.title('Model Performance Comparison')
plt.ylabel('Score')
plt.xticks(rotation=45)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 9. Error Analysis

In [ ]:
# Analyze misclassified images
misclassified_indices = np.where(y_test != y_pred)[0]
print(f"Number of misclassified images: {len(misclassified_indices)}")
print(f"Error rate: {len(misclassified_indices) / len(y_test) * 100:.2f}%")

# Visualize misclassified images
if len(misclassified_indices) > 0:
    num_display = min(16, len(misclassified_indices))
    fig, axes = plt.subplots(4, 4, figsize=(15, 15))
    
    for idx, ax in enumerate(axes.flat):
        if idx >= num_display:
            ax.axis('off')
            continue
        
        i = misclassified_indices[idx]
        ax.imshow(X_test[i])
        
        true_label = 'Good' if y_test[i] == 1 else 'Bad'
        pred_label = 'Good' if y_pred[i] == 1 else 'Bad'
        confidence = y_pred_proba[i][y_pred[i]]
        
        ax.set_title(f'True: {true_label}\nPred: {pred_label} ({confidence:.2%})',
                    color='red', fontsize=10)
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()

## 10. Save and Export

In [ ]:
# Save model in different formats
# TensorFlow SavedModel format
model.save('models/wool_classifier_saved_model')

# Save as TFLite (for mobile deployment)
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()
with open('models/wool_classifier.tflite', 'wb') as f:
    f.write(tflite_model)

print("Models exported successfully!")